# The kernel trick in code
A linear SVM fails on circular data; the RBF and degree-2 polynomial kernels separate it without any new columns.

In [1]:
import numpy as np
import plotly.graph_objects as go
from sklearn.datasets import make_circles
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

## 1. The data: 100 points in two circles

In [2]:
# factor = radius of the inner circle relative to the outer one; noise = spread
X, y = make_circles(n_samples=100, factor=0.1, noise=0.1, random_state=5)
print(X.shape, np.bincount(y))      # 100 points, 2 columns, 50 per class

colours = np.array(["#4C78A8", "#E45756"])     # class 0 blue, class 1 red
go.Figure(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers", marker=dict(color=colours[y]))).update_layout(
    template="simple_white", width=500, height=500, yaxis_scaleanchor="x").show()

(100, 2) [50 50]


## 2. A linear SVM

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

linear = SVC(kernel="linear")          # a straight-line boundary
linear.fit(X_train, y_train)
y_pred = linear.predict(X_test)
print("linear kernel accuracy:", accuracy_score(y_test, y_pred))

linear kernel accuracy: 0.55


In [4]:
def plot_decision_boundary(model, X, y, title=""):
    # predict every point of a fine grid, colour the grid by the prediction, then draw the data on top
    xs = np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 200)
    ys = np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 200)
    XX, YY = np.meshgrid(xs, ys)
    Z = model.predict(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    fig = go.Figure(go.Heatmap(x=xs, y=ys, z=Z, showscale=False, opacity=0.3,
                               colorscale=[[0, "#4C78A8"], [1, "#E45756"]]))
    fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers", marker=dict(color=colours[y])))
    fig.update_layout(template="simple_white", width=500, height=500, title=title, yaxis_scaleanchor="x")
    fig.show()

plot_decision_boundary(linear, X, y, "linear kernel")

## 3. Lifting the data into 3D
Apply exp(-x²) to each coordinate and add the two results: points near the centre get a large z.

In [5]:
X_new = np.exp(-(X ** 2))        # exp(-x1^2) and exp(-x2^2) for every point
z = X_new.sum(axis=1)            # the height: exp(-x1^2) + exp(-x2^2)
print("z of the ring:  ", z[y == 0].min().round(2), "to", z[y == 0].max().round(2))
print("z of the centre:", z[y == 1].min().round(2), "to", z[y == 1].max().round(2))

fig = go.Figure(go.Scatter3d(x=X[:, 0], y=X[:, 1], z=z, mode="markers", marker=dict(color=colours[y], size=4)))
fig.update_layout(width=600, height=500, scene=dict(xaxis_title="x1", yaxis_title="x2", zaxis_title="z"))
fig.show()

z of the ring:   1.01 to 1.56
z of the centre: 1.89 to 2.0


In [6]:
# the same transformation seen in 2D: the two new columns before they are added
go.Figure(go.Scatter(x=X_new[:, 0], y=X_new[:, 1], mode="markers", marker=dict(color=colours[y]))).update_layout(
    template="simple_white", width=500, height=500, xaxis_title="exp(-x1^2)", yaxis_title="exp(-x2^2)").show()

## 4. The RBF kernel: same two columns, no new features

In [7]:
rbf = SVC(kernel="rbf")
rbf.fit(X_train, y_train)
print("RBF kernel accuracy:", accuracy_score(y_test, rbf.predict(X_test)))
plot_decision_boundary(rbf, X, y, "RBF kernel")

RBF kernel accuracy: 1.0


## 5. The polynomial kernel: degree 3 (the default), then degree 2

In [8]:
for degree in [3, 2]:
    poly = SVC(kernel="poly", degree=degree)
    poly.fit(X_train, y_train)
    print(f"polynomial kernel, degree {degree}: accuracy {accuracy_score(y_test, poly.predict(X_test))}")
    plot_decision_boundary(poly, X, y, f"polynomial kernel, degree {degree}")

polynomial kernel, degree 3: accuracy 0.45


polynomial kernel, degree 2: accuracy 1.0


In [9]:
# degree 3 with coef0=1 adds the lower-degree terms (such as x1^2 + x2^2) back
poly31 = SVC(kernel="poly", degree=3, coef0=1)
poly31.fit(X_train, y_train)
print("degree 3, coef0=1 accuracy:", accuracy_score(y_test, poly31.predict(X_test)),
      "support vectors:", len(poly31.support_vectors_))

degree 3, coef0=1 accuracy: 1.0 support vectors: 8


## 6. Why it is a trick: a kernel is a dot product in a bigger space
For two points a and b, the degree-2 kernel (a · b)² equals the dot product of the 3 new features (x1², √2·x1·x2, x2²), computed without building them.

In [10]:
def phi(p):
    # the explicit degree-2 features of one 2D point
    return np.array([p[0] ** 2, np.sqrt(2) * p[0] * p[1], p[1] ** 2])

a, b = X[0], X[1]
print("kernel (a . b)^2     :", round((a @ b) ** 2, 6))
print("phi(a) . phi(b)      :", round(phi(a) @ phi(b), 6))

kernel (a . b)^2     : 0.002641
phi(a) . phi(b)      : 0.002641


## 7. Number of support vectors

In [11]:
for name, model in [("linear", linear), ("rbf", rbf)]:
    print(name, "support vectors:", len(model.support_vectors_), "of", len(X_train))

linear support vectors: 76 of 80
rbf support vectors: 13 of 80


In [12]:
# Points inside the margin or misclassified have y * f(x) < 1; these must be support vectors
signed = linear.decision_function(X_train) * (2 * y_train - 1)
print("linear: misclassified", (signed < 0).sum(), "| inside margin or wrong side", (signed < 1).sum(), "of", len(X_train))

linear: misclassified 30 | inside margin or wrong side 76 of 80


## 8. Playground
The Dash app in `app.py` lets you change the kernel, C, gamma and degree. Run `python app.py` in this folder and open http://127.0.0.1:8050, or set `RUN_APP = True` to show it here.

In [13]:
RUN_APP = False
if RUN_APP:
    from app import app
    app.run(jupyter_mode="inline")